In [1]:

import warnings

from plotly.subplots import make_subplots


warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from hummingbot_api_client import HummingbotAPIClient

load_dotenv()

print("✅ Hummingbot Status Reporter initialized")
print("🤖 Using official hummingbot-api-client")

✅ Hummingbot Status Reporter initialized
🤖 Using official hummingbot-api-client


In [2]:
import os

# 📊 Fetch Portfolio State and Active Bots
portfolio_state = None
active_bots_data = None
portfolio_value = 0.0
top_balances = []
active_bots = []
api_connected = False
host = os.getenv("HUMMINGBOT_API_HOST", "localhost") # Change if Hummingbot API is on a different host

print("📊 Fetching data from Hummingbot API...")

async with HummingbotAPIClient(base_url=f"http://{host}:8000") as client:
    try:
        # Get portfolio state
        try:
            portfolio_state = await client.portfolio.get_state()
            if portfolio_state and 'master_account' in portfolio_state:
                api_connected = True
                print("✅ Portfolio state retrieved")

                # Process portfolio data
                master_account = portfolio_state['master_account']

                for exchange, assets in master_account.items():
                    if isinstance(assets, list):
                        for asset_info in assets:
                            token = asset_info.get('token', 'Unknown')
                            units = float(asset_info.get('units', 0))
                            price = float(asset_info.get('price', 0))
                            value = float(asset_info.get('value', 0))

                            # Only include significant balances (> $1)
                            if units > 0 and value >= 1.0:
                                top_balances.append({
                                    'asset': token,
                                    'exchange': exchange,
                                    'units': units,
                                    'price': price,
                                    'value': value
                                })
                                portfolio_value += value

                # Sort by value descending
                top_balances.sort(key=lambda x: x['value'], reverse=True)
                print(f"✅ Portfolio: ${portfolio_value:,.2f} in {len(top_balances)} positions")

        except Exception as e:
            print(f"⚠️  Portfolio state: {e}")

        # Get active bots status
        try:
            active_bots_data = await client.bot_orchestration.get_active_bots_status()
            if active_bots_data and active_bots_data.get('status') == 'success':
                print("✅ Active bots data retrieved")

                bots_data = active_bots_data.get('data', {})

                for bot_name, bot_info in bots_data.items():
                    if isinstance(bot_info, dict) and 'status' in bot_info:
                        bot_status = bot_info.get('status', 'Unknown')
                        performance = bot_info.get('performance', {})

                        # Extract performance metrics
                        total_pnl = 0.0
                        total_volume = 0.0
                        strategies_count = len(performance)

                        for strategy_name, strategy_perf in performance.items():
                            if isinstance(strategy_perf, dict) and 'performance' in strategy_perf:
                                perf_data = strategy_perf['performance']
                                total_pnl += float(perf_data.get('global_pnl_quote', 0))
                                total_volume += float(perf_data.get('volume_traded', 0))

                        active_bots.append({
                            'name': bot_name,
                            'status': bot_status,
                            'strategies_count': strategies_count,
                            'total_pnl': total_pnl,
                            'total_volume': total_volume
                        })

                print(f"✅ Active bots: {len(active_bots)} running")

        except Exception as e:
            print(f"⚠️  Active bots: {e}")

    except Exception as e:
        print(f"❌ API Connection failed: {e}")

📊 Fetching data from Hummingbot API...
✅ Portfolio state retrieved
✅ Portfolio: $29,010.99 in 3 positions
✅ Active bots data retrieved
✅ Active bots: 1 running


In [5]:
import json

snapshot = performance  # tu dict con el performance

with open("performance_log.jsonl", "a") as f:
    f.write(json.dumps(snapshot) + "\n")
